In [0]:
# Constantes del laboratorio — usar en todos los notebooks de tareas
CATALOG       = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing"
SOURCE_PATH   = f"{VOLUME_PATH}/sesion09"

BRONZE_TABLE  = f"{CATALOG}.{SCHEMA_BRONZE}.transacciones_retail_raw"
SILVER_TABLE  = f"{CATALOG}.{SCHEMA_SILVER}.transacciones_clean"
GOLD_TABLE    = f"{CATALOG}.{SCHEMA_GOLD}.ventas_por_tienda"

print(f"Source  : {SOURCE_PATH}")
print(f"Bronze  : {BRONZE_TABLE}")
print(f"Silver  : {SILVER_TABLE}")
print(f"Gold    : {GOLD_TABLE}")

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg,
    max as spark_max, round as spark_round
)

silver_count = dbutils.jobs.taskValues.get(
    taskKey="silver_transform",
    key="silver_count",
    default=0,
    debugValue=33
)
print(f"Procesando {silver_count} registros desde silver")

df_silver = spark.table(SILVER_TABLE)

df_gold = (
    df_silver
    .groupBy("tienda_id", "categoria")
    .agg(
        count("order_id").alias("num_transacciones"),
        spark_round(spark_sum("total"), 2).alias("ingresos_total"),
        spark_round(avg("total"), 2).alias("ticket_promedio"),
        spark_sum("cantidad").alias("unidades_vendidas"),
        spark_max("fecha").alias("ultima_venta")
    )
)

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

gold_count = spark.table(GOLD_TABLE).count()
print(f"Gold generado: {gold_count} filas (combinaciones tienda x categoria) -> {GOLD_TABLE}")
display(spark.table(GOLD_TABLE).orderBy("tienda_id", "categoria"))